In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
print("Connected!")

Connected!


In [2]:
# Row counts per table
tables = ['transactions', 'merchants', 'users', 'states', 'time_dim']
for t in tables:
    df = pd.read_sql(f"SELECT COUNT(*) as count FROM {t}", engine)
    print(f"{t}: {df['count'][0]:,} rows")

transactions: 500,000 rows
merchants: 500 rows
users: 5,000 rows
states: 20 rows
time_dim: 17,521 rows


In [3]:
# Fraud rate overview
df = pd.read_sql("""
    SELECT 
        is_fraud,
        COUNT(*) as count,
        ROUND(AVG(amount)::numeric, 2) as avg_amount
    FROM transactions
    GROUP BY is_fraud
""", engine)
print(df)

   is_fraud   count  avg_amount
0     False  486730     8517.19
1      True   13270    12239.45


In [4]:
# Top merchant categories by volume
df = pd.read_sql("""
    SELECT m.category, COUNT(*) as txn_count, ROUND(SUM(t.amount)::numeric, 2) as revenue
    FROM transactions t
    JOIN merchants m ON t.merchant_id = m.merchant_id
    GROUP BY m.category
    ORDER BY revenue DESC
""", engine)
print(df)

             category  txn_count       revenue
0         Electronics      57115  2.426998e+09
1  Travel & Transport      44071  3.345248e+08
2       Entertainment      61769  3.119078e+08
3   Retail & Shopping      59033  2.991820e+08
4          Healthcare      52853  2.670069e+08
5           Education      45129  2.275122e+08
6             Fashion      44310  2.247012e+08
7   Utilities & Bills      51077  1.293142e+08
8       Food & Dining      47887  4.903307e+07
9           Groceries      36756  3.780965e+07
